In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
df = pd.read_csv('../data/Wildlife_Export_2023_2025.csv', encoding='latin-1')

C:\Users\Dheick\AppData\Local\Temp\ipykernel_45324\3854085828.py:1: DtypeWarning: Columns (0: AIRPORT_LATITUDE, 1: AIRPORT_LONGITUDE, 2: BIRD_BAND_NUMBER, 3: LUPDATE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/Wildlife_Export_2023_2025.csv', encoding='latin-1')


### Stripping the Column Headers

In [3]:
df.columns = df.columns.str.strip()

### Getting More Information About the Data
* There are 65,774 records for the time period between 2023 & 2025
* This dataset has 102 columns, much of this data can probably be dropped.

In [4]:
df.shape

(65774, 102)

### Checking the datatype for all columns
<u>Issues with the datatypes</u>
* `INCIDENT_DATE` is a string
* `TIME` is a string
* `NR_INJURIES` is a float
* `NR_FATALITIES` is a float

In [5]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 65774 entries, 0 to 65773
Data columns (total 102 columns):
 #    Column                 Non-Null Count  Dtype  
---   ------                 --------------  -----  
 0    INDX_NR                65774 non-null  int64  
 1    INCIDENT_DATE          65774 non-null  str    
 2    INCIDENT_MONTH         65774 non-null  int64  
 3    INCIDENT_YEAR          65774 non-null  int64  
 4    TIME                   65774 non-null  str    
 5    TIME_OF_DAY            34364 non-null  str    
 6    AIRPORT_ID             65774 non-null  str    
 7    AIRPORT                65774 non-null  str    
 8    AIRPORT_LATITUDE       53792 non-null  object 
 9    AIRPORT_LONGITUDE      53789 non-null  object 
 10   RUNWAY                 65774 non-null  str    
 11   STATE                  53792 non-null  str    
 12   FAAREGION              53792 non-null  str    
 13   LOCATION               11853 non-null  str    
 14   ENROUTE_STATE          1048 non-null   str    


### Checking for Null Data
- `null_pct` calculates the percentage of missing values for each column.
- The print statement filters to columns where more than **50%** of values are null.
- Columns exceeding this threshold will be candidates for removal.

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(null_pct[null_pct > 50])

### Converting `INCIDENT_DATE` to `datetime`
* This code will convert `INCIDENT_DATE` to `datetime` which will provide for more robust data. 

In [ ]:
df['INCIDENT_DATE'] = pd.to_datetime(df['INCIDENT_DATE'])

---

### Comparing TIME and TIME_OF_DAY
- `TIME` reports zero nulls, while `TIME_OF_DAY` is missing 47% of its values.
    - However, inspecting the top 20 rows via `head(20)` reveals visible blanks in `TIME` — suggesting these are empty strings rather than true nulls, since the column's `str` dtype won't flag them with `isna()`.

In [8]:
print(df['TIME'].isna().sum(), df['TIME'].isna().mean())
print(df['TIME'].isnull().sum(), df['TIME'].isnull().mean())
print(df['TIME_OF_DAY'].isna().sum(), df['TIME_OF_DAY'].isna().mean())

0 0.0
0 0.0
31410 0.47754431842369327


In [7]:
df['TIME'].head(20)

0          
1     13:49
2      7:15
3     14:10
4      8:50
5     10:12
6     10:30
7      8:50
8     10:30
9     20:30
10    10:17
11    12:30
12         
13    14:37
14     0:22
15     8:12
16         
17         
18         
19         
Name: TIME, dtype: str

When I query the `TIME` column for empty strings, it returns `0` - suggesting that what I visually see as empty strings are not actually empty.

In [9]:
print((df['TIME'] == '').sum())

0


This code strips whitespace from the `TIME` column and replaces any blank values with `NaN`. Querying `TIME` for `NaN` returns 21,865 records, confirming that a significant portion of the dataset is missing time values.

In [10]:
df['TIME'] = df['TIME'].str.strip().replace('', np.nan)

In [11]:
print(df['TIME'].isna().sum())

21865


The missing values in `TIME_OF_DAY` are detectable by `isna()`, resulting in 31,410 values missing in the dataset for time of day.

In [12]:
print(df['TIME_OF_DAY'].isna().sum())

31410


---

### Exploration of `TIME` and `TIME_OF_DAY`
There appears to be an inverse relationship between `TIME` and `TIME_OF_DAY`: when one is populated, the other may be missing. To reconcile these columns, I'll define a `time_to_minutes()` function that splits the `TIME` string on the colon and converts it to total elapsed minutes (hours × 60 + minutes). From there, I can derive a unified `TIME_OF_DAY` column that captures the time of day for every event.
- [ ] Confirm if this inverse relationship exists
- [ ] Determin if `TIME` can inform `TIME_OF_DAY`

In [13]:
def time_to_minutes(t):
    if pd.isna(t) or t == '':
        return np.nan
    h, m = t.split(':')
    return int(h) * 60 + int(m)

df['TIME_MINUTES'] = df['TIME'].apply(time_to_minutes)

In [14]:
df[['TIME_MINUTES','TIME_OF_DAY']].head(20)

,TIME_MINUTES,TIME_OF_DAY
0,NaN,Night
1,829.0,Day
2,435.0,NaN
3,850.0,NaN
4,530.0,NaN
5,612.0,NaN
6,630.0,NaN
7,530.0,NaN
8,630.0,NaN
9,1230.0,NaN


---

### `NR_INJURIES` and `NR_FATALITIES`
Both columns are stored as `float` but should be `int` — injury and fatality counts are 
whole numbers by definition. The high null rate (99.96% and 99.99% respectively) reflects 
that the vast majority of wildlife strikes cause no harm, not that the data is poorly 
collected. The records that do contain values are rare and worth retaining for analysis.

In [ ]:
df['NR_INJURIES'] = df['NR_INJURIES'].fillna(0).astype(int)

In [ ]:
df['NR_FATALITIES'] = df['NR_FATALITIES'].fillna(0).astype(int)

### Checking for Duplicates
* There are no duplicate rows

In [ ]:
df.duplicated().sum()

---

### Aditional Observations About the Data
* `INCIDENT_YEAR` The number of bird strikes is increasing year-over-year from 2023 to 2025.
* `INCIDENT_MONTH` Most bird strikes are recorded in September, August, October and July. There seems to be a significant difference between the warmer and colder months.
* `STATE` The top 5 states are TX, FL, CA, CO and TN. This seems to track with where the busy airports are (perhaps I can bring some data in regarding airport activity). Kentuky made the top 10.
* `FAAREGION` There seems to be some duplication in the FAAREGION data. I will need to strip this column to normalize it.
* `AIRPORT` UKNOWN is by far the largest airport, I am curious why that is the case. My guess is that it because many of the bird strikes are from observation of evidence of a bird strike but a lack of clarity where it happened. The other airports make sense because they are large market hubs.
* `OPERATOR` Again, UNKNOWN is by far the largest operator followed by the other popular airlines. I do find it interesting that business beats Delta and United, as those are very popular airlines.
* `PHASE_OF_FLIGHT` This is interesting because it describes at what point a plane is most likely to have a bird strike. I will need to supliment my understanding of these phases of flight because I do not know what makes them distinct.




In [ ]:
df['INCIDENT_YEAR'].value_counts()

In [ ]:
df['INCIDENT_MONTH'].value_counts()

In [ ]:
df['STATE'].value_counts().head(10)

In [ ]:
df['FAAREGION'].value_counts().head(10)

In [15]:
df['FAAREGION'] = df['FAAREGION'].str.strip().replace('', np.nan)

In [ ]:
df['FAAREGION'].value_counts().head(10)

In [ ]:
df['AIRPORT'].value_counts().head(10)

In [ ]:
df['OPERATOR'].value_counts().head(10)

In [ ]:
df['NR_FATALITIES'].value_counts()

In [ ]:
df['PHASE_OF_FLIGHT'].value_counts().head(10)

In [ ]:
df['TIME_OF_DAY'].value_counts().head(10)

In [ ]:
df['TIME_OF_DAY'].isna().sum()

In [ ]:
df['TIME_OF_DAY'].value_counts(dropna=False)

In [ ]:
df['TIME'].head(10)

In [ ]:
df['INGESTED_OTHER'].value_counts()

In [ ]:
df['ENG_1_POS'].value_counts()

### Exploring the Danger of Bird Strikes
The original dataset has two fields to display number of injuries and deaths due to FAA bird strikes. I will create a new dataframe to just include columns that add context to the strikes that caused injury or death called `df_human_impact`.
* On January 1, 2024, there was an incident where 3 individuals died after striking a Cackling Goose.
* There are 22 injuries recorded between spanning between 2023-2025, one of them being a white-tailed deer.


In [ ]:
df_human_impact = df[['INCIDENT_DATE','AIRCRAFT','NR_INJURIES','NR_FATALITIES','SPECIES','ENROUTE_STATE']]

In [ ]:
df_human_impact[df_human_impact['NR_FATALITIES']>0]

In [ ]:
df_human_impact[df_human_impact['NR_INJURIES']>0].sort_values('NR_INJURIES', ascending=False)

### Exploration of Birds
**Most Frequently Observed Species** "Unknown bird" and "Unknown bird - small" are the two most commonly recorded strike species. This likely reflects the difficulty of identification mid-air or the severity of the collision leaving little physical evidence. The remaining species are common birds found throughout the country.

`BIRD_BAND_NUMBER` This column initially seemed promising for cross-referencing external data, but reliable additional resources are limited. This column will likely be dropped from further analysis.

` NUM_STRUCK` - **Surprising Finding** The frequency of multi-bird strikes (11-100 and 100+) was unexpected. This challenges the assumption that this would not happen often.

`SIZE` Strike distribution by size follows an expected pattern: small birds are most frquently struck, followed by medium, then large. 

In [ ]:
df['SPECIES'].value_counts().head(10)

In [ ]:
df['BIRD_BAND_NUMBER'].value_counts().head()

In [ ]:
df['NUM_STRUCK'].value_counts()

In [ ]:
df['SIZE'].value_counts()

### Species and Size: Grouping BIrds for Analysis
Segmenting birds by size category (small, medium, large) opens up several useful lines of questions:

* **Most common species by size** - Which bird species is most frequently involved in strikes within each size category?
* **Strike frequency by size** - Are certain size categories struck more often than others?
* **Impact severity by size** - Does bird size correlate with the degree of damage or operational effect on the aircraft?
* **Size vs. flight phase** - Are birds of different sizes more likely to be struck during specific phases of flight?

In [ ]:
df_LGBIRD = df[df['SIZE'] == 'Large']

In [ ]:
df_MDBIRD = df[df['SIZE'] == 'Medium']

In [ ]:
df_SMBIRD = df[df['SIZE'] == 'Small']

### Top 10 Large, Medium, and Small Birds
#### Explanation
Using `value_counts().head(10)` to identify the top 10 species involved in strikes within each size category.

#### Observations
- **Large:** The category includes non-bird wildlife — skunks, coyotes, jackrabbits, and deer — alongside the top entry, `Unknown bird - large`. The presence of ground animals is unexpected given typical airport perimeter fencing, suggesting these strikes may be concentrated at smaller, less-secured airports.
- **Medium:** Predominantly birds, with `Unknown bird - medium` as the most frequent entry.
- **Small:** Same pattern as medium — `Unknown bird - small` leads the category.

In [ ]:
df_LGBIRD['SPECIES'].value_counts().head(10)

In [ ]:
df_MDBIRD['SPECIES'].value_counts().head(10)

In [ ]:
df_SMBIRD['SPECIES'].value_counts().head(10)

### Exploration of Airplane and Flight Data
* `AC_MASS` is a float ranging from 1 - 5

In [ ]:
df['AC_MASS'].value_counts()

In [ ]:
df['AC_MASS'].describe()

In [ ]:
df['AIRCRAFT'].value_counts()

In [ ]:
df['HEIGHT'].describe()